# NUSMods Module Review Sentiment Dataset

to scrape module reviews from NUSMods Disqus. using gpt-oss-20b review 1-5 sentiment score + short explanation.

added checkpoint system so we don't gotta restart.

## 0. Setup & Config


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PATH = "/content/drive/My Drive/01_Now/01_NUS/ORBITAL/Analysis/"
os.makedirs(DRIVE_PATH, exist_ok=True)

FORUM_NAME = "nusmods-prod"          # disqus shortname
REVIEW_COUNT_THRESHOLD = 5
MIN_WORDS = 5                        # for dropping reviews shorter than X no. of words.
CHECKPOINT_INTERVAL = 50             # to save labelling progress
EARLIEST_AY_START_YEAR = 2014        # how far back we'll be reading reviews from

RAW_JSON_OUT      = os.path.join(DRIVE_PATH, "nusmods_reviews_raw.json")
RAW_CSV_OUT        = os.path.join(DRIVE_PATH, "nusmods_reviews_raw.csv")
POSTS_CHECKPOINT   = os.path.join(DRIVE_PATH, "nusmods_posts_checkpoint.json")
LABEL_CHECKPOINT   = os.path.join(DRIVE_PATH, "labelling_checkpoint.json")
TRAIN_CSV_OUT       = os.path.join(DRIVE_PATH, "train_reviews.csv")
TEST_CSV_OUT        = os.path.join(DRIVE_PATH, "test_reviews.csv")

print("Config loaded. Files will be saved under:", DRIVE_PATH)


Mounted at /content/drive
Config loaded. Files will be saved under: /content/drive/My Drive/01_Now/01_NUS/ORBITAL/Analysis/


## 1. Scrape Reviews from Disqus

pulls every thread (=module) and every post (=review/comment) using Disquis API

In [2]:
import requests, json, csv, time, re
from bs4 import BeautifulSoup
import pandas as pd

def _get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("DISQUIS_API_KEY")
        if key:
            print("API key loaded from Colab Secrets.")
            return key
    except Exception:
        pass
    key = os.environ.get("DISQUS_API_KEY", "")
    if key:
        print("API key loaded from environment variable.")
        return key
    raise RuntimeError(
        "No API key found."
    )

DISQUS_API_KEY = _get_api_key()
LIMIT = 100   # Disqus max per request
DELAY = 1.0


API key loaded from Colab Secrets.


In [3]:
def verify_forum():
    resp = requests.get(
        "https://disqus.com/api/3.0/forums/details.json",
        params={"api_key": DISQUS_API_KEY, "forum": FORUM_NAME},
        timeout=15
    )
    data = resp.json()
    code_ = data.get("code", -1)
    if code_ in (2, 5):
        raise RuntimeError(
            f"API key rejected (Disqus code {code_}).\n"
            "  Go to https://disqus.com/api/applications/ -> your app\n"
            "  Copy the PUBLIC KEY (not Secret Key) and update your Colab Secret."
        )
    if code_ == 18:
        raise RuntimeError(
            f"Forum not found (Disqus code {code_}).\n"
            "  Check the shortname: open nusmods.com, right-click -> View Page Source,\n"
            "  search for 'disqus_shortname' to find the correct value."
        )
    if code_ != 0:
        raise RuntimeError(f"Disqus error {code_}: {data.get('response', data)}")
    info = data["response"]
    print(f"Forum verified: '{info['name']}' | shortname='{info['id']}' | posts={info.get('posts','?')}")
    return info


def fetch_all(endpoint, label, extra=None):
    """Downloads every page from a Disqus list endpoint. Returns a flat list of items."""
    url = f"https://disqus.com/api/3.0/{endpoint}"
    items = []
    cursor = None
    page = 0
    print(f"\nFetching {label}...")
    while True:
        params = {"api_key": DISQUS_API_KEY, "forum": FORUM_NAME, "limit": LIMIT, "order": "asc"}
        if extra:
            params.update(extra)
        if cursor:
            params["cursor"] = cursor

        for attempt in range(5):
            try:
                resp = requests.get(url, params=params, timeout=30)
            except requests.RequestException as e:
                wait = 2 ** attempt
                print(f"  Network error (attempt {attempt+1}): {e}. Retrying in {wait}s...")
                time.sleep(wait)
                continue
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 30))
                print(f"  Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue
            if resp.status_code != 200:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
            break
        else:
            raise RuntimeError("Max retries exceeded - aborting.")

        body = resp.json()
        if body.get("code", 0) != 0:
            raise RuntimeError(f"Disqus error {body['code']}: {body.get('response')}")

        batch = body.get("response", [])
        items.extend(batch)
        page += 1
        print(f"  Page {page:>4}: +{len(batch):>4}  |  total {len(items):>6}", end="\r")

        cursor_data = body.get("cursor", {})
        if cursor_data.get("hasNext"):
            cursor = cursor_data["next"]
            time.sleep(DELAY)
        else:
            print(f"\n  Done: {label}: {len(items)} items fetched across {page} pages.")
            break
    return items


def clean_html(html_text):
    if not html_text:
        return ""
    text = BeautifulSoup(html_text, "html.parser").get_text(separator=" ")
    text = re.sub(r"Module review by.*?:", "", text)
    text = re.sub(r"Taken in AY\d+/\d+ Sem \d+", "", text)
    text = re.sub(r"Module review also posted here:.*", "", text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def module_code_from_title(title):
    """'CS1010 Programming Methodology' -> 'CS1010'"""
    if not title:
        return "UNKNOWN"
    first = title.strip().split()[0]
    return first if re.match(r"^[A-Z]{2,4}\d{4}[A-Z]?$", first) else "UNKNOWN"


In [4]:
print("=" * 60)
print(" NUSMods Disqus Scraper")
print("=" * 60)

verify_forum()

threads = fetch_all("forums/listThreads.json", "threads")
thread_map = {}
for t in threads:
    thread_map[t["id"]] = {
        "module_code": module_code_from_title(t.get("title", "")),
        "module_title": t.get("title", "Unknown"),
        "module_link": t.get("link", ""),
    }
known = sum(1 for v in thread_map.values() if v["module_code"] != "UNKNOWN")
print(f"  Thread map: {len(thread_map)} threads, {known} with valid module codes.")

raw_posts = fetch_all("forums/listPosts.json", "posts", extra={"include": "approved"})
with open(POSTS_CHECKPOINT, "w", encoding="utf-8") as f:
    json.dump(raw_posts, f)
print(f"  Checkpoint saved -> {POSTS_CHECKPOINT}")

records = []
for post in raw_posts:
    tid = post.get("thread")
    info = thread_map.get(tid, {"module_code": "UNKNOWN", "module_title": "Unknown", "module_link": ""})
    raw_msg = post.get("message", "") or ""
    records.append({
        "comment_id": post.get("id"),
        "thread_id": tid,
        "module_code": info["module_code"],
        "module_title": info["module_title"],
        "module_link": info["module_link"],
        "author_name": post.get("author", {}).get("name", "Anonymous"),
        "date": post.get("createdAt"),
        "raw_message": raw_msg,
        "message": clean_html(raw_msg),
        "likes": post.get("likes", 0),
        "dislikes": post.get("dislikes", 0),
        "parent_comment_id": post.get("parent"),
    })

with open(RAW_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)
if records:
    with open(RAW_CSV_OUT, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)

df_reviews = pd.DataFrame(records)
df_reviews = df_reviews[
    (df_reviews["module_code"] != "UNKNOWN") & (df_reviews["message"].str.strip() != "")
].reset_index(drop=True)

print("\n" + "=" * 60)
print(f" Done. {len(df_reviews)} usable reviews across {df_reviews['module_code'].nunique()} modules.")
print(f"    Saved -> {RAW_JSON_OUT} | {RAW_CSV_OUT}")
print("=" * 60)
df_reviews[["module_code", "module_title", "date", "message"]].head(5)


 NUSMods Disqus Scraper
Forum verified: 'NUSMods' | shortname='nusmods-prod' | posts=?

Fetching threads...
  Page  207: +  45  |  total  20645
  Done: threads: 20645 items fetched across 207 pages.
  Thread map: 20645 threads, 18996 with valid module codes.

Fetching posts...
  Page   97: +  16  |  total   9616
  Done: posts: 9616 items fetched across 97 pages.
  Checkpoint saved -> /content/drive/My Drive/01_Now/01_NUS/ORBITAL/Analysis/nusmods_posts_checkpoint.json

 Done. 9530 usable reviews across 2070 modules.
    Saved -> /content/drive/My Drive/01_Now/01_NUS/ORBITAL/Analysis/nusmods_reviews_raw.json | /content/drive/My Drive/01_Now/01_NUS/ORBITAL/Analysis/nusmods_reviews_raw.csv


,module_code,module_title,date,message
0,CS3216,CS3216 Software Development on Evolving Platforms,2014-06-29T23:43:01,Awesomeness!
1,CS1020E,CS1020E Data Structures and Algorithms I,2014-07-06T17:08:51,This looks interesting :). Anybody has any tip...
2,CS1020E,CS1020E Data Structures and Algorithms I,2014-07-06T19:07:57,"Hey Bhavesh, sure thing, you can find more inf..."
3,ACC1002X,ACC1002X Financial Accounting,2014-07-06T20:20:30,Definitely a course to take for people who wan...
4,CS3216,CS3216 Software Development on Evolving Platforms,2014-07-07T10:04:11,For those who are thinking of taking CS3216 in...


## 2. Filter Low-Content Reviews

drop comments that are under `min words`.

In [5]:
before = len(df_reviews)
df_reviews["word_count"] = df_reviews["message"].apply(lambda t: len(t.split()))
df_reviews = df_reviews[df_reviews["word_count"] >= MIN_WORDS].reset_index(drop=True)
after = len(df_reviews)

print(f"Dropped {before - after} reviews under {MIN_WORDS} words ({before} -> {after} remaining).")


Dropped 546 reviews under 5 words (9530 -> 8984 remaining).


## 3. Train/Test Split

In [6]:
review_counts = df_reviews.groupby("module_code").size().rename("review_count")
df_reviews = df_reviews.merge(review_counts, on="module_code", how="left")

train_modules = set(review_counts[review_counts < REVIEW_COUNT_THRESHOLD].index)
test_modules = set(review_counts[review_counts >= REVIEW_COUNT_THRESHOLD].index)

df_reviews["split"] = df_reviews["module_code"].apply(lambda m: "train" if m in train_modules else "test")

print(f"Modules: {len(train_modules)} in train (<{REVIEW_COUNT_THRESHOLD} reviews), "
      f"{len(test_modules)} in test (>={REVIEW_COUNT_THRESHOLD} reviews)")
print(f"Reviews: {(df_reviews['split']=='train').sum()} train rows, "
      f"{(df_reviews['split']=='test').sum()} test rows")


Modules: 1551 in train (<5 reviews), 481 in test (>=5 reviews)
Reviews: 2662 train rows, 6322 test rows


## 4. Enrich Modules w. NUSMods v2 API Metadata

For each module, try current academic year first, then go backwards.

In [7]:
from datetime import datetime

def current_academic_year():
    now = datetime.now()
    start_year = now.year if now.month >= 8 else now.year - 1
    return f"{start_year}-{start_year+1}"

def academic_year_candidates(latest_ay, earliest_start_year=EARLIEST_AY_START_YEAR):
    """AY strings from latest_ay walking backwards to earliest_start_year."""
    latest_start = int(latest_ay.split("-")[0])
    years = []
    y = latest_start
    while y >= earliest_start_year:
        years.append(f"{y}-{y+1}")
        y -= 1
    return years

LATEST_AY = current_academic_year()
AY_CANDIDATES = academic_year_candidates(LATEST_AY)
print(f"Latest AY: {LATEST_AY}. Will fall back through {len(AY_CANDIDATES)} years if needed "
      f"(down to {AY_CANDIDATES[-1]}).")

_module_meta_cache = {}

def fetch_module_metadata(module_code):
    """Try latest AY first, then walk backwards for discontinued modules.
    Returns a dict; found=False if the module was never found in any year."""
    if module_code in _module_meta_cache:
        return _module_meta_cache[module_code]

    for ay in AY_CANDIDATES:
        url = f"https://api.nusmods.com/v2/{ay}/modules/{module_code}.json"
        try:
            resp = requests.get(url, timeout=15)
        except requests.RequestException:
            continue
        if resp.status_code == 200:
            data = resp.json()
            meta = {
                "found": True,
                "found_ay": ay,
                "description": data.get("description", ""),
                "moduleCredit": data.get("moduleCredit", ""),
                "department": data.get("department", ""),
                "faculty": data.get("faculty", ""),
                "workload": data.get("workload", ""),
                "prerequisite": data.get("prerequisite", ""),
                "preclusion": data.get("preclusion", ""),
            }
            _module_meta_cache[module_code] = meta
            return meta
        time.sleep(0.1)

    meta = {"found": False, "found_ay": None, "description": "", "moduleCredit": "",
             "department": "", "faculty": "", "workload": "", "prerequisite": "", "preclusion": ""}
    _module_meta_cache[module_code] = meta
    return meta


Latest AY: 2025-2026. Will fall back through 12 years if needed (down to 2014-2015).


In [8]:
from tqdm import tqdm

unique_modules = df_reviews["module_code"].unique().tolist()
print(f"Fetching metadata for {len(unique_modules)} unique modules...")

meta_records = []
for code_ in tqdm(unique_modules, desc="Module metadata"):
    meta = fetch_module_metadata(code_)
    meta_records.append({"module_code": code_, **meta})

df_meta = pd.DataFrame(meta_records)
not_found = (~df_meta["found"]).sum()
print(f"Metadata found for {len(df_meta) - not_found}/{len(df_meta)} modules "
      f"({not_found} not found in any year - likely a mis-parsed thread title).")

df_reviews = df_reviews.merge(df_meta, on="module_code", how="left")
df_reviews.head()


Fetching metadata for 2032 unique modules...


Module metadata: 100%|██████████| 2032/2032 [26:19<00:00,  1.29it/s]

Metadata found for 2012/2032 modules (20 not found in any year - likely a mis-parsed thread title).


,comment_id,thread_id,module_code,module_title,module_link,author_name,date,raw_message,message,likes,...,split,found,found_ay,description,moduleCredit,department,faculty,workload,prerequisite,preclusion
0,1504826420,2874104284,CS1020E,CS1020E Data Structures and Algorithms I,http://nusmods.com/modules/CS1020E/reviews,Bhavesh .R,2014-07-06T17:08:51,<p>This looks interesting :). Anybody has any ...,This looks interesting :). Anybody has any tip...,0,...,test,True,2021-2022,This module is the second part of a three-part...,4,Computer Science,Computing,"[2, 1, 1, 3, 3]",CS1010E or its equivalent,"CS1020, CS2020, CS2030, CS2040, CS2040C"
1,1504826421,2874104284,CS1020E,CS1020E Data Structures and Algorithms I,http://nusmods.com/modules/CS1020E/reviews,Yangshun,2014-07-06T19:07:57,"<p>Hey Bhavesh, sure thing, you can find more ...","Hey Bhavesh, sure thing, you can find more inf...",0,...,test,True,2021-2022,This module is the second part of a three-part...,4,Computer Science,Computing,"[2, 1, 1, 3, 3]",CS1010E or its equivalent,"CS1020, CS2020, CS2030, CS2040, CS2040C"
2,1504826396,2874104252,ACC1002X,ACC1002X Financial Accounting,http://nusmods.com/modules/ACC1002X/reviews,Toh Weiqing,2014-07-06T20:20:30,<p>Definitely a course to take for people who ...,Definitely a course to take for people who wan...,1,...,test,True,2020-2021,The course provides an introduction to financi...,4,Accounting,NUS Business School,,,Students who have passed CS1304 or EC3212 or B...
3,1504826381,2874104237,CS3216,CS3216 Software Development on Evolving Platforms,http://nusmods.com/modules/CS3216/reviews,Yangshun,2014-07-07T10:04:11,<p>For those who are thinking of taking CS3216...,For those who are thinking of taking CS3216 in...,3,...,test,True,2025-2026,"In this course, students will practice softwar...",5,Computer Science,Computing,"[2, 1, 0, 8, 2]",If undertaking an Undergraduate DegreeTHEN( mu...,
4,1504826391,2874104250,CS3241,CS3241 Computer Graphics,http://nusmods.com/modules/CS3241/reviews,Yangshun,2014-07-07T22:37:03,<p>CS3241 is the module you have to take if yo...,CS3241 is the module you have to take if you d...,2,...,test,True,2025-2026,This course teaches some graphics hardware dev...,4,Computer Science,Computing,"[2, 1, 0, 3, 3]",If undertaking an Undergraduate DegreeTHEN( mu...,


## 5. Load Local LLM (`unsloth/gpt-oss-20b`)

Same install + load pattern as your Reddit sentiment notebook — runs locally on the Colab GPU, no API key needed for this part.

In [9]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers<=0.23.0 trl==0.22.2 unsloth unsloth_zoo

In [10]:
%%capture
!uv pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth-zoo
!uv pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth
!pip uninstall -y pyarrow datasets
!pip install --no-cache-dir "pyarrow==17.0.0" "datasets==3.6.0"

In [11]:
from unsloth import FastLanguageModel
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b-unsloth-bnb-4bit",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


<frozen importlib._bootstrap>:488: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C header, got 72 from PyObject
<frozen importlib._bootstrap>:488: RuntimeWarning: pyarrow.lib._Tabular size changed, may indicate binary incompatibility. Expected 24 from C header, got 32 from PyObject
<frozen importlib._bootstrap>:488: RuntimeWarning: pyarrow.lib.Table size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

## 6. Sentiment Labelling Function (1-5 scale)

Same JSON-structured prompting pattern as your Reddit notebook's `get_sentiment`, adapted to a 1-5 score + explanation with an explicit rubric so the model has a consistent reference point.

In [12]:
import json as _json
import re as _re

SENTIMENT_RUBRIC = """1 = very negative (student strongly disliked the module / had a bad experience)
2 = negative (more downsides than upsides)
3 = neutral / mixed (balanced, or not clearly positive or negative)
4 = positive (more upsides than downsides)
5 = very positive (student strongly enjoyed / recommends the module)"""

def get_sentiment(text):
    prompt = f"""You are a sentiment labeling assistant for NUS module reviews.

Review:
\"\"\"{text}\"\"\"

Rate the sentiment of this review on a 1-5 scale:
{SENTIMENT_RUBRIC}

Respond with **ONLY** valid JSON. Nothing else. <brief reason> should be maximum 20 words.

Format: {{"sentiment_score": <1-5 integer>, "explanation": "<brief reason>"}}
"""

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        reasoning_effort="medium",
    ).to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        temperature=0.0,
    )

    gen_ids = output[0][inputs["input_ids"].shape[-1]:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    try:
        result = _json.loads(decoded)
    except _json.JSONDecodeError:
        match = _re.search(r'\{[^{}]*"sentiment_score"[^{}]*\}', decoded)
        if match:
            try:
                result = _json.loads(match.group(0))
            except _json.JSONDecodeError:
                return None, f"error: invalid JSON: {decoded[:100]}"
        else:
            return None, f"error: invalid JSON: {decoded[:100]}"

    if "sentiment_score" not in result or "explanation" not in result:
        return None, "error: missing required fields"

    try:
        score = int(result["sentiment_score"])
    except (TypeError, ValueError):
        return None, f"error: non-integer score: {result.get('sentiment_score')}"

    if score < 1 or score > 5:
        return None, f"error: score out of range: {score}"

    return score, result["explanation"]


## 7. Run Labelling (Resumable)

In [13]:
BATCH_SIZE = 8

def get_sentiment_batch(texts):
    prompts = []
    for text in texts:
        prompt = f"""You are a sentiment labeling assistant for NUS module reviews.

Review:
\"\"\"{text}\"\"\"

Rate the sentiment of this review on a 1-5 scale:
1 = very negative (student strongly disliked the module / had a bad experience)
2 = negative (more downsides than upsides)
3 = neutral / mixed (balanced, or not clearly positive or negative)
4 = positive (more upsides than downsides)
5 = very positive (student strongly enjoyed / recommends the module)

Respond with ONLY valid JSON. <brief reason> should be maximum 20 words.

Format: {{"sentiment_score": <1-5 integer>, "explanation": "<brief reason>"}}
"""
        messages = [{"role": "user", "content": prompt}]
        prompts.append(tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ))

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096,
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )

    results_batch = []
    for i, output in enumerate(outputs):
        input_len = inputs["input_ids"][i].shape[-1]
        gen_ids = output[input_len:]
        decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        try:
            result = json.loads(decoded)
        except json.JSONDecodeError:
            match = re.search(r'\{[^{}]*"sentiment_score"[^{}]*\}', decoded)
            if match:
                try:
                    result = json.loads(match.group(0))
                except json.JSONDecodeError:
                    results_batch.append((None, f"error: invalid JSON: {decoded[:100]}"))
                    continue
            else:
                results_batch.append((None, f"error: invalid JSON: {decoded[:100]}"))
                continue

        if "sentiment_score" not in result or "explanation" not in result:
            results_batch.append((None, "error: missing fields"))
            continue

        try:
            score = int(result["sentiment_score"])
        except (TypeError, ValueError):
            results_batch.append((None, "error: non-integer score"))
            continue

        if score < 1 or score > 5:
            results_batch.append((None, f"error: score out of range: {score}"))
            continue

        results_batch.append((score, result["explanation"]))

    return results_batch


def load_checkpoint():
    if os.path.exists(LABEL_CHECKPOINT):
        with open(LABEL_CHECKPOINT, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_checkpoint(results_dict):
    with open(LABEL_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(results_dict, f)

def is_successfully_labelled(cid):
    r = results.get(cid)
    return r is not None and r.get("sentiment_score") is not None


results = load_checkpoint()
print(f"Loaded {len(results)} previously labelled reviews from checkpoint.")

to_label = df_reviews[~df_reviews["comment_id"].astype(str).apply(is_successfully_labelled)]
print(f"{len(to_label)} reviews remaining to label this run.")

time_begin = time.time()
since_last_save = 0
rows = list(to_label.iterrows())

for batch_start in tqdm(range(0, len(rows), BATCH_SIZE), desc="Labelling"):
    batch_rows = rows[batch_start : batch_start + BATCH_SIZE]
    texts = [row["message"] for _, row in batch_rows]
    cids  = [str(row["comment_id"]) for _, row in batch_rows]

    scored = get_sentiment_batch(texts)

    for cid, (score, explanation) in zip(cids, scored):
        results[cid] = {"sentiment_score": score, "explanation": explanation}
        since_last_save += 1

    if since_last_save >= CHECKPOINT_INTERVAL:
        save_checkpoint(results)
        since_last_save = 0

save_checkpoint(results)
print(f"Total labelling time this run: {(time.time()-time_begin)/60:.2f} minutes")
print(f"Total labelled overall: {len(results)} / {len(df_reviews)}")


Loaded 2897 previously labelled reviews from checkpoint.
7083 reviews remaining to label this run.


Labelling:   0%|          | 0/886 [04:20<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 840.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 533.81 MiB is free. Including non-PyTorch memory, this process has 14.04 GiB memory in use. Of the allocated memory 13.52 GiB is allocated by PyTorch, and 70.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)